# YOLO26m 独立正式完整微调

本 Notebook 使用官方 YOLO26m 和固定数据划分，输出与 YOLO26s 完全隔离。正式训练完成后会在固定 test 集评估，并与当前 YOLO26s 基线比较精度和速度；任何产物都不会自动部署。

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import json
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'training').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise RuntimeError('无法定位项目根目录')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)

PROJECT_ROOT: D:\dev\projects\yolo_train


## 环境与真实 YOLO26m 结构检查

Kernel 必须使用 `Python (YOLO26 Project)` 或 `.venv-yolo26\Scripts\python.exe`。检测到 YOLO-Master 或 CPU 时立即停止。

In [2]:
import torch
import ultralytics
from ultralytics import YOLO

print('Python:', sys.version.split()[0])
print('Ultralytics:', ultralytics.__version__, ultralytics.__file__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if '.venv-yolo26' not in str(Path(ultralytics.__file__).resolve()):
    raise RuntimeError('Kernel 不是项目独立 YOLO26 环境，请切换后重启 Kernel')
if not torch.cuda.is_available():
    raise RuntimeError('正式 YOLO26m 训练要求 CUDA')

Python: 3.13.9
Ultralytics: 8.4.55 D:\dev\projects\yolo_train\.venv-yolo26\Lib\site-packages\ultralytics\__init__.py
CUDA: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [3]:
from training.audit_dataset import audit

DATASET_DIR = PROJECT_ROOT / 'data/processed'
DATA_YAML = DATASET_DIR / 'data.yaml'
audit_report = audit(DATASET_DIR)
if audit_report.get('errors'):
    raise RuntimeError(audit_report['errors'])
print(json.dumps(audit_report['splits'], ensure_ascii=False, indent=2))

{
  "train": {
    "images": 2816,
    "class_objects": {
      "crack": 170,
      "water": 847,
      "leak": 1154
    },
    "negative_images": 708,
    "objects_per_image": {
      "0": 708,
      "1": 2067,
      "2": 30,
      "3": 3,
      "4": 5,
      "5": 3
    }
  },
  "valid": {
    "images": 251,
    "class_objects": {
      "crack": 18,
      "water": 85,
      "leak": 97
    },
    "negative_images": 53,
    "objects_per_image": {
      "0": 53,
      "1": 196,
      "2": 2
    }
  },
  "test": {
    "images": 251,
    "class_objects": {
      "crack": 18,
      "water": 84,
      "leak": 102
    },
    "negative_images": 53,
    "objects_per_image": {
      "0": 53,
      "1": 194,
      "2": 3,
      "4": 1
    }
  }
}


## 正式训练配置

8GB 显存默认从 `batch=1`、`imgsz=768` 开始。3 epochs/640 兼容性预检和 1 epoch/768 显存预检只写入 smoke 目录，不得部署。

In [4]:
MODEL_PATH = PROJECT_ROOT / 'yolo26m.pt'
EPOCHS = 150
IMAGE_SIZE = 768
BATCH = 1
DEVICE = 0
SEED = 42
PATIENCE = 25
RESUME = False

OUTPUT_PROJECT = PROJECT_ROOT / 'models/trained/yolo26m/formal'
RUN_NAME = 'yolo26m_full_768_formal'
RUN_DIR = OUTPUT_PROJECT / RUN_NAME
LAST_PT = RUN_DIR / 'weights/last.pt'
BEST_PT = RUN_DIR / 'weights/best.pt'
METRICS_JSON = PROJECT_ROOT / 'models/metrics/yolo26m_full_768_internal_metrics.json'
BASELINE_JSON = PROJECT_ROOT / 'models/metrics/yolo26s_full_768_internal_metrics.json'

if not MODEL_PATH.is_file():
    raise FileNotFoundError(f'缺少官方权重: {MODEL_PATH}')
if RESUME and not LAST_PT.is_file():
    raise FileNotFoundError(f'续训权重不存在: {LAST_PT}')
print('正式训练目录:', RUN_DIR)

正式训练目录: D:\dev\projects\yolo_train\models\trained\yolo26m\formal\yolo26m_full_768_formal


In [5]:
model = YOLO(str(LAST_PT if RESUME else MODEL_PATH))
head = model.model.model[-1]
if not getattr(head, 'end2end', False) or getattr(head, 'reg_max', None) != 1:
    raise RuntimeError('当前运行时没有构建真实 YOLO26 端到端检测头')
if not hasattr(head, 'one2one_cv2') or not hasattr(head, 'one2one_cv3'):
    raise RuntimeError('YOLO26m 缺少 one-to-one 检测分支')
print('检测头:', type(head).__name__, 'end2end=', head.end2end, 'reg_max=', head.reg_max)
print('参数量:', f'{sum(p.numel() for p in model.model.parameters()) / 1e6:.2f}M')

train_results = model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH,
    device=DEVICE, workers=0, optimizer='MuSGD', patience=PATIENCE,
    seed=SEED, deterministic=True, amp=True, pretrained=True,
    close_mosaic=15, project=str(OUTPUT_PROJECT), name=RUN_NAME,
    exist_ok=True, plots=True, save=True, save_period=10, resume=RESUME,
)

检测头: Detect end2end= True reg_max= 1
参数量: 21.90M
Ultralytics 8.4.55  Python-3.13.9 torch-2.13.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\dev\projects\yolo_train\data\processed\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\dev\projects\yolo_train\yolo26m.pt, momentum=0.937, mosaic=1.0, multi_s

KeyboardInterrupt: 

In [ ]:
if not BEST_PT.is_file():
    raise FileNotFoundError(f'训练没有生成 best.pt: {BEST_PT}')
results_csv = RUN_DIR / 'results.csv'
with results_csv.open(encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
map50 = [float(row['metrics/mAP50(B)']) for row in rows]
map50_95 = [float(row['metrics/mAP50-95(B)']) for row in rows]
best_index = max(range(len(rows)), key=lambda i: map50_95[i])
actual_epochs = len(rows)
if max(map50, default=0.0) < 0.50:
    raise RuntimeError('正式训练未通过最低指标检查，禁止评估或部署')
print('实际训练轮次:', actual_epochs)
print('最佳轮次:', rows[best_index]['epoch'])
print('最佳 valid mAP50:', map50[best_index])
print('最佳 valid mAP50-95:', map50_95[best_index])

In [ ]:
import matplotlib.pyplot as plt

with results_csv.open(encoding='utf-8') as handle:
    curve_rows = [{key.strip(): value for key, value in row.items()} for row in csv.DictReader(handle) if row.get('epoch')]

def curve(column):
    return [float(row[column]) for row in curve_rows]

plot_epochs = [int(float(row['epoch'])) for row in curve_rows]
plot_map50 = curve('metrics/mAP50(B)')
plot_map50_95 = curve('metrics/mAP50-95(B)')
best_plot_index = max(range(len(curve_rows)), key=lambda i: plot_map50_95[i])
best_plot_epoch = plot_epochs[best_plot_index]
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
axes[0, 0].plot(plot_epochs, curve('train/box_loss'), label='train box')
axes[0, 0].plot(plot_epochs, curve('val/box_loss'), label='valid box')
axes[0, 0].set(title='Box Loss', xlabel='Epoch', ylabel='Loss')
axes[0, 1].plot(plot_epochs, curve('train/cls_loss'), label='train cls')
axes[0, 1].plot(plot_epochs, curve('val/cls_loss'), label='valid cls')
axes[0, 1].set(title='Classification Loss', xlabel='Epoch', ylabel='Loss')
axes[1, 0].plot(plot_epochs, curve('metrics/precision(B)'), label='Precision')
axes[1, 0].plot(plot_epochs, curve('metrics/recall(B)'), label='Recall')
axes[1, 0].set(title='Precision / Recall', xlabel='Epoch', ylabel='Score', ylim=(0, 1.02))
axes[1, 1].plot(plot_epochs, plot_map50, label='mAP50')
axes[1, 1].plot(plot_epochs, plot_map50_95, label='mAP50-95')
axes[1, 1].scatter([best_plot_epoch], [plot_map50_95[best_plot_index]], color='red', zorder=5, label='best mAP50-95')
axes[1, 1].set(title='Validation mAP', xlabel='Epoch', ylabel='mAP', ylim=(0, 1.02))
for ax in axes.flat:
    ax.axvline(best_plot_epoch, color='gray', linestyle='--', linewidth=1, alpha=0.6)
    ax.grid(True, alpha=0.25)
    ax.legend()
figure_path = RUN_DIR / 'training_curves.png'
fig.suptitle(f'YOLO26m Training Results | best epoch={best_plot_epoch}', fontsize=15)
fig.savefig(figure_path, dpi=160, bbox_inches='tight')
plt.show()
print('训练曲线已保存:', figure_path)

## 固定 test 集最终评估及 YOLO26s 基线比较

只使用 `best.pt`。调参过程中以 valid 为准，正式配方锁定后才运行本 Cell。

In [ ]:
from training.common import TARGET_NAMES

best_model = YOLO(str(BEST_PT))
metrics = best_model.val(
    data=str(DATA_YAML), split='test', imgsz=IMAGE_SIZE, batch=1, device=DEVICE,
    workers=0, plots=True, project=str(PROJECT_ROOT / 'models/metrics/evaluation_runs'),
    name='yolo26m_full_768_internal', exist_ok=True,
)
current_metrics = {
    'source': 'formal_yolo26m_full_768_notebook',
    'model_path': str(BEST_PT), 'dataset': str(DATA_YAML), 'split': 'test',
    'configured_epochs': EPOCHS, 'actual_epochs': actual_epochs,
    'precision': float(metrics.box.mp), 'recall': float(metrics.box.mr),
    'map50': float(metrics.box.map50), 'map50_95': float(metrics.box.map),
    'per_class_map50_95': {name: float(value) for name, value in zip(TARGET_NAMES, metrics.box.maps)},
    'speed_ms': {key: float(value) for key, value in metrics.speed.items()},
    'model_size_mb': BEST_PT.stat().st_size / (1024 ** 2),
    'created_at': datetime.now(timezone.utc).isoformat(),
}
METRICS_JSON.parent.mkdir(parents=True, exist_ok=True)
METRICS_JSON.write_text(json.dumps(current_metrics, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(current_metrics, ensure_ascii=False, indent=2))

if BASELINE_JSON.is_file():
    baseline = json.loads(BASELINE_JSON.read_text(encoding='utf-8'))
    comparison = {
        key: {'yolo26s': baseline[key], 'yolo26m': current_metrics[key], 'delta': current_metrics[key] - baseline[key]}
        for key in ('precision', 'recall', 'map50', 'map50_95')
    }
    comparison['inference_ms'] = {
        'yolo26s': baseline['speed_ms']['inference'],
        'yolo26m': current_metrics['speed_ms']['inference'],
        'delta': current_metrics['speed_ms']['inference'] - baseline['speed_ms']['inference'],
    }
    print('YOLO26s / YOLO26m 对比:')
    print(json.dumps(comparison, ensure_ascii=False, indent=2))
    accuracy_gate = comparison['map50_95']['delta'] >= 0.01 and current_metrics['recall'] >= baseline['recall']
    speed_gate = current_metrics['speed_ms']['inference'] <= 15.0
    print('精度门槛:', accuracy_gate, '| 速度门槛:', speed_gate)

In [ ]:
EXPORT_ONNX = True
exported = None
if EXPORT_ONNX:
    exported = best_model.export(format='onnx', imgsz=IMAGE_SIZE, device=DEVICE)
    print('ONNX:', exported)

manifest = {
    'training_line': 'yolo26m_full_768_formal',
    'mainline_isolated': True, 'best_pt': str(BEST_PT),
    'onnx': str(exported) if exported else None, 'metrics_json': str(METRICS_JSON),
    'official_ultralytics_version': ultralytics.__version__,
    'end2end_nms_free': True, 'automatically_deployed': False,
}
manifest_path = RUN_DIR / 'notebook_model_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifest, ensure_ascii=False, indent=2))